# Chess Board Warping — Batch Generator (v3 • no summaries)

This notebook/script processes folders of raw images and exports a warped, square, top-down
view of a chessboard for each image. It uses a single calibration frame (`calib.*`) per folder
to determine a fixed perspective transform (homography) and then applies that same homography to
all other images in the folder. This guarantees that every output is aligned exactly the same way
as the calibration frame.

Key points:
- No CSVs are written (no global `summary.csv`, no per-folder `manifest.csv`).
- One calibration image named `calib.*` must exist in each image folder.
- The calibration step detects chessboard corners, estimates a homography H from image → board
  coordinates, composes it with a margin/scale transform, and stores it as `h0`.
- For every image, we skip any per-image tracking/RANSAC and always apply `h0`.
- Outputs are PNG images suffixed with `_warp.png` written under `OUTPUT_ROOT` mirrored by input
  folder structure.


In [1]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, Optional, List, Dict

import cv2
import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

print('Python', sys.version)
print('OpenCV', cv2.__version__)
print('NumPy', np.__version__)


Python 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
OpenCV 4.12.0
NumPy 2.3.5


In [2]:
BOARD_SIZE_PX = 1024
INPUT_TARGET_LONG_EDGE = 1600
BOARD_MARGIN_SQUARES = 1.7


@dataclass
class BoardProjection:
    """Lightweight representation of a warped test and the square polygons.

    Attributes:
        warped_board: The output test image in canonical top-down view of size BOARD_SIZE_PX×BOARD_SIZE_PX.
        squares_px:   Array of shape (8, 8, 4, 2) with the four corner points (x, y) of each original-image
                      square polygon (A1..H8) mapped back into the source image's pixel space.
    """
    warped_board: np.ndarray
    squares_px: np.ndarray


@dataclass
class _BoardRectification:
    """Full rectification result used internally during detection/calibration.

    Attributes:
        mode:       Text label describing how rectification was obtained (e.g., 'corners', 'calib_h0').
        h:          3×3 homography mapping input image → canonical test image with margins.
        h_inv:      Inverse of `h`, mapping canonical test coords → input image.
        warped:     The warped test image of size BOARD_SIZE_PX×BOARD_SIZE_PX.
        squares_px: The 8×8 grid of square polygons in input-image coordinates.
        input_gray: Grayscale version of the (possibly resized) input used for detection.
    """
    mode: str
    h: np.ndarray
    h_inv: np.ndarray
    warped: np.ndarray
    squares_px: np.ndarray
    input_gray: np.ndarray


@dataclass
class CalibrationCache:
    """Data kept from the calibration frame.

    Only the calibration homography `h0` is stored. All subsequent images will be warped using
    exactly this matrix to ensure identical alignment across the dataset.

    Attributes:
        h0: 3×3 homography from input image to canonical test image.
    """
    h0: np.ndarray


def _resize_to_fixed_long_edge(bgr: np.ndarray, target_long: int = INPUT_TARGET_LONG_EDGE) -> Tuple[np.ndarray, float]:
    """Resize image so that its longer side equals `target_long` pixels.

    Uses area interpolation when downscaling and linear when upscaling.

    Args:
        bgr:         Input color image (BGR).
        target_long: Desired length of the longer side after resizing.

    Returns:
        (resized_bgr, scale): The resized image and the scale factor applied to width/height.
    """
    if bgr is None or bgr.size == 0:
        raise ValueError('Input image is empty or failed to load.')
    h, w = bgr.shape[:2]
    long_edge = max(h, w)
    if long_edge == 0:
        raise ValueError('Invalid image with zero dimension.')
    if long_edge == target_long:
        return bgr, 1.0
    s = float(target_long) / float(long_edge)
    new_w, new_h = int(round(w * s)), int(round(h * s))
    interp = cv2.INTER_AREA if s < 1.0 else cv2.INTER_LINEAR
    return cv2.resize(bgr, (new_w, new_h), interpolation=interp), s


def _px_step_and_offset(board_size_px: int, margin_squares: float):
    """Compute pixel step and offset for the canonical test image with margins.

    The canonical test is an (8×8) grid centered inside the output image, padded on all sides by
    `margin_squares` squares. This function returns the pixel size of one square (step) and the
    top-left pixel offset (offset) where the A1 square starts.
    """
    step = board_size_px / (8.0 + 2.0 * margin_squares)
    offset = step * margin_squares
    return step, offset


def _S_margin_matrix(board_size_px: int, margin_squares: float) -> np.ndarray:
    """Build the similarity transform that maps test coordinates → pixel coords with margins.

    The corner detector yields coordinates in a test space (units of squares). We compose the
    chessboard homography with this scale/translate transform so the final warp occupies a centered
    region with margins in the output image.
    """
    step, offset = _px_step_and_offset(board_size_px, margin_squares)
    S = np.array([[step, 0.0, offset], [0.0, step, offset], [0.0, 0.0, 1.0]], dtype=np.float32)
    return S


def _compute_square_polygons(h_inv: np.ndarray, margin_squares: float) -> np.ndarray:
    """Return polygons of all 64 squares back-projected into the input image.

    For each square in the canonical test image, take its 4-corner rectangle in pixel coordinates
    and transform the corners by `h_inv` to get the corresponding quadrilateral in the original image.
    """
    step, offs = _px_step_and_offset(BOARD_SIZE_PX, margin_squares)
    polys = np.zeros((8, 8, 4, 2), dtype=np.float32)
    for r in range(8):
        for c in range(8):
            x0 = offs + c * step
            y0 = offs + r * step
            quad = np.array([[x0, y0], [x0 + step, y0], [x0 + step, y0 + step], [x0, y0 + step]], dtype=np.float32)
            polys[r, c] = cv2.perspectiveTransform(quad[None, ...], h_inv.astype(np.float32))[0]
    return polys


def _try_find_corners(gray: np.ndarray, pattern_size: Tuple[int, int]) -> Optional[np.ndarray]:
    """Detect inner chessboard corners using OpenCV's robust or legacy methods.

    Prefers `findChessboardCornersSB` (more robust) when available, otherwise falls back to
    `findChessboardCorners` with reasonable flags.

    Returns:
        Corners as (N, 1, 2) float32 array or None if detection fails.
    """
    if hasattr(cv2, 'findChessboardCornersSB'):
        out = cv2.findChessboardCornersSB(gray, pattern_size, cv2.CALIB_CB_EXHAUSTIVE | cv2.CALIB_CB_ACCURACY)
        if isinstance(out, tuple) and out[0] and out[1] is not None:
            return out[1].astype(np.float32)
        if not isinstance(out, tuple) and out is not None:
            return out.astype(np.float32)
    # Fall back to classic API with commonly used flags for robustness.
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_FILTER_QUADS
    ok, corners = cv2.findChessboardCorners(gray, pattern_size, flags)
    return corners.astype(np.float32) if ok and corners is not None else None


def _solve_homography_corners(bgr_pre: np.ndarray, pattern_size: Tuple[int, int]) -> Optional[_BoardRectification]:
    """Estimate test homography from a single image by detecting chessboard corners.

    Steps:
    1) Detect and sub-pixel refine inner chessboard corners in the preprocessed image.
    2) Build a corresponding set of object points in a test coordinate system (1 unit = 1 square).
    3) Solve a homography H_board that maps image points → test coordinates via RANSAC.
    4) Compose with S (scale/translate) to produce `h_total` that maps to a padded, square output.
    5) Warp the image and precompute per-square polygons back in input-image space.
    """
    gray = cv2.cvtColor(bgr_pre, cv2.COLOR_BGR2GRAY)
    corners = _try_find_corners(gray, pattern_size)
    if corners is None:
        return None
    cv2.cornerSubPix(gray, corners, winSize=(5, 5), zeroZone=(-1, -1),
                     criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 60, 1e-4))
    imgp = corners.reshape(-1, 2).astype(np.float32)
    nx, ny = pattern_size
    objp = np.array([[x + 1, y + 1] for y in range(ny) for x in range(nx)], dtype=np.float32)
    h_board, _ = cv2.findHomography(imgp, objp, method=cv2.RANSAC)
    if h_board is None or not np.isfinite(h_board).all():
        return None
    # Compose with a scale/translate matrix that lays the test into the output image with margins.
    S_m = _S_margin_matrix(BOARD_SIZE_PX, BOARD_MARGIN_SQUARES)
    h_total = (S_m @ h_board).astype(np.float32)
    h_inv = np.linalg.inv(h_total)
    warped = cv2.warpPerspective(bgr_pre, h_total, (BOARD_SIZE_PX, BOARD_SIZE_PX), flags=cv2.INTER_LINEAR)
    squares_px = _compute_square_polygons(h_inv, BOARD_MARGIN_SQUARES)
    return _BoardRectification(mode='corners', h=h_total, h_inv=h_inv, warped=warped, squares_px=squares_px,
                               input_gray=gray)


def detect_and_rectify_board(bgr: np.ndarray, pattern_size: Tuple[int, int] = (7, 7)) -> Optional[_BoardRectification]:
    """Detect chessboard in a single image and return rectification artifacts.

    This is used on the calibration frame to compute `h0`.
    """
    if bgr is None or bgr.size == 0:
        raise ValueError('empty image')
    bgr_pre, _ = _resize_to_fixed_long_edge(bgr)
    return _solve_homography_corners(bgr_pre, pattern_size)


def build_calibration_cache(rect: _BoardRectification) -> CalibrationCache:
    """Create the calibration cache from a rectified calibration frame.

    Only the homography is kept since all subsequent frames will be warped using this matrix.
    """
    return CalibrationCache(h0=rect.h.copy().astype(np.float32))


def detect_and_rectify_board_with_cache(bgr: np.ndarray, cache: CalibrationCache) -> Optional[_BoardRectification]:
    """Apply the calibration homography to an image to obtain its warped test view.

    Notes:
    - This deliberately ignores per-image updates (e.g., optical flow or re-estimation).
    - Guarantees identical alignment to the calibration image.
    """
    if bgr is None or bgr.size == 0:
        raise ValueError('empty image')
    bgr_pre, _ = _resize_to_fixed_long_edge(bgr)
    # Enforce using the calibration homography (h0) for all images to ensure
    # consistent warps matching the calibration frame. We intentionally skip
    # any per-image tracking/RANSAC updates here.
    h = cache.h0.astype(np.float32)
    h_inv = np.linalg.inv(h)
    warped = cv2.warpPerspective(bgr_pre, h, (BOARD_SIZE_PX, BOARD_SIZE_PX), flags=cv2.INTER_LINEAR)
    return _BoardRectification(mode='calib_h0', h=h, h_inv=h_inv, warped=warped,
                               squares_px=_compute_square_polygons(h_inv, BOARD_MARGIN_SQUARES),
                               input_gray=cv2.cvtColor(bgr_pre, cv2.COLOR_BGR2GRAY))


def get_board_projection_with_cache(bgr: np.ndarray, cache: CalibrationCache) -> Optional[BoardProjection]:
    """Convenience wrapper that returns only the public projection object.

    This is useful for downstream code that needs just the warped test image and square polygons.
    """
    rect = detect_and_rectify_board_with_cache(bgr, cache)
    if rect is None:
        return None
    return BoardProjection(warped_board=rect.warped, squares_px=rect.squares_px.astype(np.float32))


In [3]:
# from pathlib import Path  # already imported above

# === EDIT THESE ===
# Paths and processing parameters. Adjust these to your environment/dataset.
INPUT_ROOTS = [
    "../../data/raw_images/game6",
]
OUTPUT_ROOT = Path("../data/processed")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}  # Image file extensions to include.
PATTERN_SIZE = (7, 7)  # Inner-corner pattern size expected by OpenCV detectors.
MAX_LONG_EDGE = 0  # Optional additional downscale for calibration image (0 disables).
EXCLUDE_CALIB_FROM_OUTPUT = True  # Whether to skip writing the calib frame to outputs.


In [4]:
def discover_image_folders(roots: List[str]) -> Dict[Path, List[Path]]:
    """Find all folders under given roots that contain at least one image.

    Returns a mapping from folder → sorted list of image paths.
    """
    result: Dict[Path, List[Path]] = {}
    for root in roots:
        root_p = Path(root)
        if not root_p.exists():
            print(f"[WARN] Root not found: {root_p}")
            continue
        for dirpath, _, filenames in os.walk(root_p):
            folder = Path(dirpath)
            imgs = [folder / f for f in filenames if Path(f).suffix.lower() in IMG_EXTS]
            imgs.sort()
            if imgs:
                result[folder] = imgs
    return result


def imread_color(path: Path) -> Optional[np.ndarray]:
    """Read an image in BGR color, returning None on failure instead of raising."""
    arr = cv2.imread(str(path))
    return arr if arr is not None and arr.size > 0 else None


def save_image(path: Path, img: np.ndarray) -> None:
    """Write an image to disk, creating parent directories if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), img)


def rel_to_any_root(p: Path, roots: List[str]) -> Path:
    """Get a path of `p` relative to any of the provided roots, if possible.

    If `p` is not under any root, the function falls back to using just the name.
    """
    for r in roots:
        try:
            return p.relative_to(Path(r))
        except Exception:
            continue
    return p.name


def find_calib_in_folder(folder: Path, images: List[Path]) -> Path:
    """Return the unique calibration image in a folder named exactly `calib.*`.

    Raises a clear error if not found or if multiple candidates exist.
    """
    cands = [p for p in images if p.stem.lower() == 'calib' and p.suffix.lower() in IMG_EXTS]
    if len(cands) == 1:
        return cands[0]
    if len(cands) == 0:
        raise RuntimeError(f"No calibration image named 'calib.*' in {folder}")
    raise RuntimeError(f"Multiple 'calib.*' images in {folder}: {cands}")


In [5]:
from collections import defaultdict


def rectify_with_cache_folder(folder: Path, imgs: List[Path], out_root: Path, roots: List[str]) -> Dict:
    """Process one folder: calibrate once, warp all images with the same homography.

    Workflow:
      1) Find and read `calib.*`.
      2) Optionally downscale it further (MAX_LONG_EDGE) to ease detection if very large.
      3) Detect corners and compute `h0` using `detect_and_rectify_board`.
      4) For each image, apply `h0` via `detect_and_rectify_board_with_cache` and save the result.

    Returns a small stats dictionary for logging/debugging.
    """
    stats = {'folder': str(folder), 'total': len(imgs), 'calibrated': False, 'ok': 0, 'fail': 0,
             'by_mode': defaultdict(int), 'samples': [], 'calibration_used': ''}
    calib_path = find_calib_in_folder(folder, imgs)
    bgr_calib = imread_color(calib_path)
    if bgr_calib is None:
        raise RuntimeError(f"Cannot read calibration image: {calib_path}")
    if MAX_LONG_EDGE and max(bgr_calib.shape[:2]) > MAX_LONG_EDGE:
        # Additional safety downscale for calibration if it's extremely large.
        s = MAX_LONG_EDGE / max(bgr_calib.shape[:2])
        bgr_calib = cv2.resize(bgr_calib, (int(bgr_calib.shape[1] * s), int(bgr_calib.shape[0] * s)),
                               interpolation=cv2.INTER_AREA)
    rect0 = detect_and_rectify_board(bgr_calib, pattern_size=PATTERN_SIZE)
    if rect0 is None:
        raise RuntimeError(f"Calibration failed on {calib_path}")
    cache = build_calibration_cache(rect0)
    stats['calibrated'] = True
    stats['calibration_used'] = str(calib_path)

    for img_path in tqdm(imgs, desc=f"Process {folder.name}"):
        if EXCLUDE_CALIB_FROM_OUTPUT and img_path.resolve() == calib_path.resolve():
            continue
        bgr = imread_color(img_path)
        if bgr is None:
            stats['fail'] += 1
            continue
        rect = detect_and_rectify_board_with_cache(bgr, cache)
        if rect is None:
            stats['fail'] += 1
            continue
        stats['ok'] += 1
        stats['by_mode'][rect.mode] += 1
        rel = rel_to_any_root(img_path.parent, roots)
        out_dir = out_root / rel
        out_name = img_path.stem + '_warp.png'
        out_path = out_dir / out_name
        save_image(out_path, rect.warped)
        if len(stats['samples']) < 3:
            stats['samples'].append(str(out_path))
    return stats


In [6]:
if not INPUT_ROOTS:
    # Nothing to do until the user specifies where raw images live.
    print('⚠️ Please edit INPUT_ROOTS in the config cell first.')
else:
    folders = discover_image_folders(INPUT_ROOTS)
    print(f'Discovered {len(folders)} folders with images.')
    for folder, imgs in folders.items():
        try:
            s = rectify_with_cache_folder(folder, imgs, OUTPUT_ROOT, INPUT_ROOTS)
            print(f"[OK] {folder} | ok={s['ok']} fail={s['fail']} calib={s['calibration_used']}")
        except Exception as e:
            print(f"[ERROR] {folder}: {e}")
    print('Done.')


Discovered 1 folders with images.


Process game6:   0%|          | 0/33 [00:00<?, ?it/s]

[OK] ..\..\data\raw_images\game6 | ok=32 fail=0 calib=..\..\data\raw_images\game6\calib.jpg
Done.
